# 1. Extraction des données (lxml)

Ce notebook extrait les paragraphes des comptes rendus de l’Assemblée nationale à partir des fichiers XML, en utilisant la bibliothèque lxml (qui a ici donné des résultats plus simples que la précédente version).


Sans doute moins robuste, moins de gestion des erreurs et exceptions, mais plus simple, lisible, et sans doute OK sur les données assemblée qui sont stables / propres.

In [ ]:
# TODO: s'assurer que le passage vers lxml est ok
# TODO: VÉRIFIER QUE LA RÉCUP ID_ACTEUR EST OK ET SUFFISANTE

# cf : DONE: on a un problème avec des ID_orateur manquants
# qui sont sout la forme <acteurRef>PA605991</acteurRef>
# en fait on peut sans doute choper un id_acteur dans tous les cas (15 et 16eme)
# à voir si reste de la logique tient


In [ ]:
# Test 1 pour avoir les titres des parties
import os
import glob
from lxml import etree
import pandas as pd

def extraire_paragraphes_lxml(fichier_xml: str) -> pd.DataFrame:
    """
    Extrait les paragraphes d'un fichier XML de compte rendu en utilisant lxml,
    en ajoutant le titre du point de niveau 1 à chaque paragraphe.
    """
    try:
        tree = etree.parse(fichier_xml)
        root = tree.getroot()
        ns = {"ns": "http://schemas.assemblee-nationale.fr/referentiel"}

        meta = {
            "UID": root.findtext("ns:uid", namespaces=ns),
            "SeanceRef": root.findtext("ns:seanceRef", namespaces=ns),
            "SessionRef": root.findtext("ns:sessionRef", namespaces=ns),
        }
        meta_tags = [
            "dateSeance",
            "dateSeanceJour",
            "numSeanceJour",
            "numSeance",
            "typeAssemblee",
            "legislature",
            "session",
            "nomFichierJo",
            "presidentSeance",
        ]
        for tag in meta_tags:
            meta[tag] = root.findtext(f".//ns:{tag}", namespaces=ns)

        rows = []
        # Variable pour stocker le titre du point de niveau 1 actuel
        current_point_niv1_title = None

        for paragraphe in root.xpath(".//ns:paragraphe", namespaces=ns):
            # Naviguer vers le <point> parent
            point = paragraphe.getparent()
            while point is not None and point.tag != f"{{{ns['ns']}}}point":
                point = point.getparent()

            # Si on a trouvé un point parent
            if point is not None:
                # Vérifier si c'est un point de niveau 1
                if point.get("nivpoint") == "1":
                    current_point_niv1_title = point.findtext("ns:texte", namespaces=ns)

            # Récupérer le point parent direct (niveau 2 ou 3)
            point_type = point.get("code_grammaire") if point is not None else None
            point_title = (
                point.findtext("ns:texte", namespaces=ns) if point is not None else None
            )

            texte_elem = paragraphe.find("ns:texte", namespaces=ns)
            texte = (
                "".join(texte_elem.itertext()).strip()
                if texte_elem is not None
                else None
            )
            stime = texte_elem.get("stime") if texte_elem is not None else None

            # Récupérer les informations de l'orateur
            orateur = paragraphe.find(".//ns:orateur", namespaces=ns)
            nom_orateur = (
                orateur.findtext("ns:nom", namespaces=ns)
                if orateur is not None
                else None
            )
            qualite_orateur = (
                orateur.findtext("ns:qualite", namespaces=ns)
                if orateur is not None
                else None
            )
            id_orateur = (
                orateur.findtext("ns:id", namespaces=ns)
                if orateur is not None
                else None
            )

            # Ajouter les informations au DataFrame
            rows.append(
                {
                    # Métadonnées de la séance
                    "UID": meta["UID"],
                    "SeanceRef": meta["SeanceRef"],
                    "SessionRef": meta["SessionRef"],
                    "dateSeance": meta["dateSeance"],
                    "dateSeanceJour": meta["dateSeanceJour"],
                    "numSeanceJour": meta["numSeanceJour"],
                    "numSeance": meta["numSeance"],
                    "typeAssemblee": meta["typeAssemblee"],
                    "legislature": meta["legislature"],
                    "session": meta["session"],
                    "nomFichierJo": meta["nomFichierJo"],
                    "presidentSeance": meta["presidentSeance"],
                    # Données du point parent :
                    "point_niv1_titre": current_point_niv1_title,
                    "point_titre": point_title,
                    "point_type": point_type,
                    # Données du paragraphe :
                    "valeur_ptsodj": paragraphe.get("valeur_ptsodj"),
                    "ordinal_prise": paragraphe.get("ordinal_prise"),
                    "ordre_absolu_seance": paragraphe.get("ordre_absolu_seance"),
                    "id_acteur": paragraphe.get("id_acteur"),
                    "id_mandat": paragraphe.get("id_mandat"),
                    "code_grammaire": paragraphe.get("code_grammaire"),
                    "code_style": paragraphe.get("code_style"),
                    "code_parole": paragraphe.get("code_parole"),
                    "id_syceron": paragraphe.get("id_syceron"),
                    "roledebat": paragraphe.get("roledebat"),
                    # Données orateur + texte :
                    "nom_orateur": nom_orateur,
                    "qualite_orateur": qualite_orateur,
                    "id_orateur": id_orateur,
                    "stime": stime,
                    "texte": texte,
                }
            )

        return pd.DataFrame(rows)

    except Exception as e:
        print(f" Erreur dans {fichier_xml} : {e}")
        return pd.DataFrame()

def traiter_dossier_compte_rendu_lxml(
    dossier_path: str, pattern: str = "*.xml"
) -> pd.DataFrame:
    """
    Traite tous les fichiers XML d'un dossier avec la fonction extraire_paragraphes_lxml().
    """
    fichiers = glob.glob(os.path.join(dossier_path, pattern))
    if not fichiers:
        print(f"Aucun fichier XML trouvé dans {dossier_path}")
        return pd.DataFrame()

    all_dfs = []
    total = len(fichiers)
    print(f"Traitement de {total} fichiers XML...\n")

    for i, fichier in enumerate(fichiers, 1):
        nom = os.path.basename(fichier)
        print(f"[{i}/{total}] {nom}...", end=" ")

        df = extraire_paragraphes_lxml(fichier)
        if not df.empty:
            print(f"{len(df)} lignes")
            all_dfs.append(df)
        else:
            print("Vide ou erreur")

    if all_dfs:
        df_final = pd.concat(all_dfs, ignore_index=True)
        print(f"\n Export terminé : {len(df_final)} lignes consolidées")
        return df_final
    else:
        return pd.DataFrame()


In [ ]:
# 2 test avec code_grammaire="TITRE_TEXTE_DISCUSSION"

def extraire_paragraphes_lxml(fichier_xml: str) -> pd.DataFrame:
    """
    Extrait les paragraphes d'un fichier XML de compte rendu en utilisant lxml,
    en ajoutant le titre du point de niveau 1 (avec code_grammaire="TITRE_TEXTE_DISCUSSION")
    à chaque paragraphe.
    """
    try:
        tree = etree.parse(fichier_xml)
        root = tree.getroot()
        ns = {"ns": "http://schemas.assemblee-nationale.fr/referentiel"}

        meta = {
            "UID": root.findtext("ns:uid", namespaces=ns),
            "SeanceRef": root.findtext("ns:seanceRef", namespaces=ns),
            "SessionRef": root.findtext("ns:sessionRef", namespaces=ns),
        }
        meta_tags = [
            "dateSeance",
            "dateSeanceJour",
            "numSeanceJour",
            "numSeance",
            "typeAssemblee",
            "legislature",
            "session",
            "nomFichierJo",
            "presidentSeance",
        ]
        for tag in meta_tags:
            meta[tag] = root.findtext(f".//ns:{tag}", namespaces=ns)

        rows = []
        # Variable pour stocker le titre du point de niveau 1 actuel
        current_point_niv1_title = None

        for paragraphe in root.xpath(".//ns:paragraphe", namespaces=ns):
            # Naviguer vers le <point> parent
            point = paragraphe.getparent()
            while point is not None and point.tag != f"{{{ns['ns']}}}point":
                point = point.getparent()

            # Si on a trouvé un point parent
            if point is not None:
                # Vérifier si c'est un point de niveau 1 ET avec le bon code_grammaire
                if point.get("nivpoint") == "1" and point.get("code_grammaire") == "TITRE_TEXTE_DISCUSSION":
                    current_point_niv1_title = point.findtext("ns:texte", namespaces=ns)

            # Récupérer le point parent direct (niveau 2 ou 3)
            point_type = point.get("code_grammaire") if point is not None else None
            point_title = (
                point.findtext("ns:texte", namespaces=ns) if point is not None else None
            )

            texte_elem = paragraphe.find("ns:texte", namespaces=ns)
            texte = (
                "".join(texte_elem.itertext()).strip()
                if texte_elem is not None
                else None
            )
            stime = texte_elem.get("stime") if texte_elem is not None else None

            # Récupérer les informations de l'orateur
            orateur = paragraphe.find(".//ns:orateur", namespaces=ns)
            nom_orateur = (
                orateur.findtext("ns:nom", namespaces=ns)
                if orateur is not None
                else None
            )
            qualite_orateur = (
                orateur.findtext("ns:qualite", namespaces=ns)
                if orateur is not None
                else None
            )
            id_orateur = (
                orateur.findtext("ns:id", namespaces=ns)
                if orateur is not None
                else None
            )

            # Ajouter les informations au DataFrame
            rows.append(
                {
                    # Métadonnées de la séance
                    "UID": meta["UID"],
                    "SeanceRef": meta["SeanceRef"],
                    "SessionRef": meta["SessionRef"],
                    "dateSeance": meta["dateSeance"],
                    "dateSeanceJour": meta["dateSeanceJour"],
                    "numSeanceJour": meta["numSeanceJour"],
                    "numSeance": meta["numSeance"],
                    "typeAssemblee": meta["typeAssemblee"],
                    "legislature": meta["legislature"],
                    "session": meta["session"],
                    "nomFichierJo": meta["nomFichierJo"],
                    "presidentSeance": meta["presidentSeance"],
                    # Données du point parent :
                    "point_niv1_titre": current_point_niv1_title,
                    "point_titre": point_title,
                    "point_type": point_type,
                    # Données du paragraphe :
                    "valeur_ptsodj": paragraphe.get("valeur_ptsodj"),
                    "ordinal_prise": paragraphe.get("ordinal_prise"),
                    "ordre_absolu_seance": paragraphe.get("ordre_absolu_seance"),
                    "id_acteur": paragraphe.get("id_acteur"),
                    "id_mandat": paragraphe.get("id_mandat"),
                    "code_grammaire": paragraphe.get("code_grammaire"),
                    "code_style": paragraphe.get("code_style"),
                    "code_parole": paragraphe.get("code_parole"),
                    "id_syceron": paragraphe.get("id_syceron"),
                    "roledebat": paragraphe.get("roledebat"),
                    # Données orateur + texte :
                    "nom_orateur": nom_orateur,
                    "qualite_orateur": qualite_orateur,
                    "id_orateur": id_orateur,
                    "stime": stime,
                    "texte": texte,
                }
            )

        return pd.DataFrame(rows)

    except Exception as e:
        print(f" Erreur dans {fichier_xml} : {e}")
        return pd.DataFrame()

def traiter_dossier_compte_rendu_lxml(
    dossier_path: str, pattern: str = "*.xml"
) -> pd.DataFrame:
    """
    Traite tous les fichiers XML d'un dossier avec la fonction extraire_paragraphes_lxml().
    """
    fichiers = glob.glob(os.path.join(dossier_path, pattern))
    if not fichiers:
        print(f"Aucun fichier XML trouvé dans {dossier_path}")
        return pd.DataFrame()

    all_dfs = []
    total = len(fichiers)
    print(f"Traitement de {total} fichiers XML...\n")

    for i, fichier in enumerate(fichiers, 1):
        nom = os.path.basename(fichier)
        print(f"[{i}/{total}] {nom}...", end=" ")

        df = extraire_paragraphes_lxml(fichier)
        if not df.empty:
            print(f"{len(df)} lignes")
            all_dfs.append(df)
        else:
            print("Vide ou erreur")

    if all_dfs:
        df_final = pd.concat(all_dfs, ignore_index=True)
        print(f"\n Export terminé : {len(df_final)} lignes consolidées")
        return df_final
    else:
        return pd.DataFrame()


In [ ]:
# Test 3 --> mais tjrs le même résultat 
def extraire_paragraphes_lxml(fichier_xml: str) -> pd.DataFrame:
    """
    Extrait les paragraphes d'un fichier XML de compte rendu en utilisant lxml,
    en associant chaque intervention au titre du point de niveau 1 correspondant
    à la partie de l'ordre du jour en cours (valeur_ptsodj).
    """
    try:
        tree = etree.parse(fichier_xml)
        root = tree.getroot()
        ns = {"ns": "http://schemas.assemblee-nationale.fr/referentiel"}

        meta = {
            "UID": root.findtext("ns:uid", namespaces=ns),
            "SeanceRef": root.findtext("ns:seanceRef", namespaces=ns),
            "SessionRef": root.findtext("ns:sessionRef", namespaces=ns),
        }
        meta_tags = [
            "dateSeance",
            "dateSeanceJour",
            "numSeanceJour",
            "numSeance",
            "typeAssemblee",
            "legislature",
            "session",
            "nomFichierJo",
            "presidentSeance",
        ]
        for tag in meta_tags:
            meta[tag] = root.findtext(f".//ns:{tag}", namespaces=ns)

        rows = []
        # Variable pour stocker le titre du point de niveau 1 actuel
        current_point_niv1_title = None
        # Variable pour stocker la dernière valeur_ptsodj de niveau 1 rencontrée
        current_valeur_ptsodj = None

        for paragraphe in root.xpath(".//ns:paragraphe", namespaces=ns):
            # Naviguer vers le <point> parent
            point = paragraphe.getparent()
            while point is not None and point.tag != f"{{{ns['ns']}}}point":
                point = point.getparent()

            # Si on a trouvé un point parent
            if point is not None:
                # Vérifier si c'est un point de niveau 1
                if point.get("nivpoint") == "1":
                    # Mettre à jour le titre et la valeur_ptsodj si c'est un nouveau point de niveau 1
                    new_valeur_ptsodj = point.get("valeur_ptsodj")
                    if new_valeur_ptsodj != current_valeur_ptsodj:
                        current_valeur_ptsodj = new_valeur_ptsodj
                        current_point_niv1_title = point.findtext("ns:texte", namespaces=ns)

            # Récupérer le point parent direct (niveau 2 ou 3)
            point_type = point.get("code_grammaire") if point is not None else None
            point_title = (
                point.findtext("ns:texte", namespaces=ns) if point is not None else None
            )

            texte_elem = paragraphe.find("ns:texte", namespaces=ns)
            texte = (
                "".join(texte_elem.itertext()).strip()
                if texte_elem is not None
                else None
            )
            stime = texte_elem.get("stime") if texte_elem is not None else None

            # Récupérer les informations de l'orateur
            orateur = paragraphe.find(".//ns:orateur", namespaces=ns)
            nom_orateur = (
                orateur.findtext("ns:nom", namespaces=ns)
                if orateur is not None
                else None
            )
            qualite_orateur = (
                orateur.findtext("ns:qualite", namespaces=ns)
                if orateur is not None
                else None
            )
            id_orateur = (
                orateur.findtext("ns:id", namespaces=ns)
                if orateur is not None
                else None
            )

            # Ajouter les informations au DataFrame
            rows.append(
                {
                    # Métadonnées de la séance
                    "UID": meta["UID"],
                    "SeanceRef": meta["SeanceRef"],
                    "SessionRef": meta["SessionRef"],
                    "dateSeance": meta["dateSeance"],
                    "dateSeanceJour": meta["dateSeanceJour"],
                    "numSeanceJour": meta["numSeanceJour"],
                    "numSeance": meta["numSeance"],
                    "typeAssemblee": meta["typeAssemblee"],
                    "legislature": meta["legislature"],
                    "session": meta["session"],
                    "nomFichierJo": meta["nomFichierJo"],
                    "presidentSeance": meta["presidentSeance"],
                    # Données du point parent :
                    "point_niv1_titre": current_point_niv1_title,
                    "point_niv1_valeur_ptsodj": current_valeur_ptsodj,
                    "point_titre": point_title,
                    "point_type": point_type,
                    # Données du paragraphe :
                    "valeur_ptsodj": paragraphe.get("valeur_ptsodj"),
                    "ordinal_prise": paragraphe.get("ordinal_prise"),
                    "ordre_absolu_seance": paragraphe.get("ordre_absolu_seance"),
                    "id_acteur": paragraphe.get("id_acteur"),
                    "id_mandat": paragraphe.get("id_mandat"),
                    "code_grammaire": paragraphe.get("code_grammaire"),
                    "code_style": paragraphe.get("code_style"),
                    "code_parole": paragraphe.get("code_parole"),
                    "id_syceron": paragraphe.get("id_syceron"),
                    "roledebat": paragraphe.get("roledebat"),
                    # Données orateur + texte :
                    "nom_orateur": nom_orateur,
                    "qualite_orateur": qualite_orateur,
                    "id_orateur": id_orateur,
                    "stime": stime,
                    "texte": texte,
                }
            )

        return pd.DataFrame(rows)

    except Exception as e:
        print(f" Erreur dans {fichier_xml} : {e}")
        return pd.DataFrame()

def traiter_dossier_compte_rendu_lxml(
    dossier_path: str, pattern: str = "*.xml"
) -> pd.DataFrame:
    """
    Traite tous les fichiers XML d'un dossier avec la fonction extraire_paragraphes_lxml().
    """
    fichiers = glob.glob(os.path.join(dossier_path, pattern))
    if not fichiers:
        print(f"Aucun fichier XML trouvé dans {dossier_path}")
        return pd.DataFrame()

    all_dfs = []
    total = len(fichiers)
    print(f"Traitement de {total} fichiers XML...\n")

    for i, fichier in enumerate(fichiers, 1):
        nom = os.path.basename(fichier)
        print(f"[{i}/{total}] {nom}...", end=" ")

        df = extraire_paragraphes_lxml(fichier)
        if not df.empty:
            print(f"{len(df)} lignes")
            all_dfs.append(df)
        else:
            print("Vide ou erreur")

    if all_dfs:
        df_final = pd.concat(all_dfs, ignore_index=True)
        print(f"\n Export terminé : {len(df_final)} lignes consolidées")
        return df_final
    else:
        return pd.DataFrame()


In [ ]:
# Test 4 :

#Remonte jusqu’au point de niveau 1 le plus proche, même s’il faut traverser plusieurs niveaux.
#Ne met à jour le titre que si le point de niveau 1 a un code_grammaire pertinent (par exemple, "TITRE_TEXTE_DISCUSSION").
#Utilise ordre_absolu_seance comme identifiant unique pour suivre les changements de partie.


def extraire_paragraphes_lxml(fichier_xml: str) -> pd.DataFrame:
    """
    Extrait les paragraphes d'un fichier XML de compte rendu de l'Assemblée nationale,
    en associant chaque intervention au titre du point de niveau 1 correspondant.
    """
    try:
        tree = etree.parse(fichier_xml)
        root = tree.getroot()
        ns = {"ns": "http://schemas.assemblee-nationale.fr/referentiel"}

        # Extraction des métadonnées (inchangé)
        meta = {
            "UID": root.findtext("ns:uid", namespaces=ns),
            "SeanceRef": root.findtext("ns:seanceRef", namespaces=ns),
            "SessionRef": root.findtext("ns:sessionRef", namespaces=ns),
        }
        meta_tags = [
            "dateSeance", "dateSeanceJour", "numSeanceJour", "numSeance",
            "typeAssemblee", "legislature", "session", "nomFichierJo", "presidentSeance"
        ]
        for tag in meta_tags:
            meta[tag] = root.findtext(f".//ns:{tag}", namespaces=ns)

        rows = []
        current_point_niv1_title = None
        current_point_niv1_ordre = None

        for paragraphe in root.xpath(".//ns:paragraphe", namespaces=ns):
            # Remonter jusqu'au point de niveau 1
            point = paragraphe.getparent()
            while point is not None and point.tag == f"{{{ns['ns']}}}point" and point.get("nivpoint") != "1":
                point = point.getparent()

            # Si on a trouvé un point de niveau 1
            if point is not None and point.get("nivpoint") == "1":
                # Mettre à jour le titre si c'est un nouveau point de niveau 1 (par ordre_absolu_seance)
                new_ordre = point.get("ordre_absolu_seance")
                if new_ordre != current_point_niv1_ordre:
                    current_point_niv1_ordre = new_ordre
                    # Optionnel : ne prendre que les points avec un code_grammaire spécifique
                    if point.get("code_grammaire") == "TITRE_TEXTE_DISCUSSION":
                        current_point_niv1_title = point.findtext("ns:texte", namespaces=ns)

            # Récupération des autres données (inchangé)
            texte_elem = paragraphe.find("ns:texte", namespaces=ns)
            texte = "".join(texte_elem.itertext()).strip() if texte_elem is not None else None
            stime = texte_elem.get("stime") if texte_elem is not None else None

            orateur = paragraphe.find(".//ns:orateur", namespaces=ns)
            nom_orateur = orateur.findtext("ns:nom", namespaces=ns) if orateur is not None else None
            qualite_orateur = orateur.findtext("ns:qualite", namespaces=ns) if orateur is not None else None
            id_orateur = orateur.findtext("ns:id", namespaces=ns) if orateur is not None else None

            rows.append({
                **meta,
                "point_niv1_titre": current_point_niv1_title,
                "point_niv1_ordre": current_point_niv1_ordre,
                "texte": texte,
                "stime": stime,
                "nom_orateur": nom_orateur,
                "qualite_orateur": qualite_orateur,
                "id_orateur": id_orateur,
            })

        return pd.DataFrame(rows)

    except Exception as e:
        print(f"Erreur dans {fichier_xml} : {e}")
        return pd.DataFrame()
    
def traiter_dossier_compte_rendu_lxml(
    dossier_path: str, pattern: str = "*.xml"
) -> pd.DataFrame:
    """
    Traite tous les fichiers XML d'un dossier avec la fonction extraire_paragraphes_lxml().
    """
    fichiers = glob.glob(os.path.join(dossier_path, pattern))
    if not fichiers:
        print(f"Aucun fichier XML trouvé dans {dossier_path}")
        return pd.DataFrame()

    all_dfs = []
    total = len(fichiers)
    print(f"Traitement de {total} fichiers XML...\n")

    for i, fichier in enumerate(fichiers, 1):
        nom = os.path.basename(fichier)
        print(f"[{i}/{total}] {nom}...", end=" ")

        df = extraire_paragraphes_lxml(fichier)
        if not df.empty:
            print(f"{len(df)} lignes")
            all_dfs.append(df)
        else:
            print("Vide ou erreur")

    if all_dfs:
        df_final = pd.concat(all_dfs, ignore_index=True)
        print(f"\n Export terminé : {len(df_final)} lignes consolidées")
        return df_final
    else:
        return pd.DataFrame()


In [ ]:
df_16 = traiter_dossier_compte_rendu_lxml("../data/raw/16-xml/compteRendu/")
# Pour le sous cas de la 16 législature : nettoyer le fichier qui n'est pas au bon endroit
# = date de 2021 et est présent dans la 15 sous le nom CRSANR5L15S2021O1N144
# TODO: check manuel
df_16 = df_16[df_16["UID"] != "CRSANR5L16S2021O1N144"]
print("suppression de CRSANR5L16S2021O1N144")

# export
df_16.to_csv("../data/interim/extract_16.csv", index=False, encoding="utf-8")
print(f"\n Export CSV : ({df_16.shape[0]} lignes)")

In [ ]:
df_16

In [ ]:
# TODO: s'assurer que l'extraction marche bien avec la 15ème législature

In [ ]:
df_15 = traiter_dossier_compte_rendu_lxml("../data/raw/15-xml/compteRendu/")
df_15.to_csv("../data/interim/extract_15.csv", index=False, encoding="utf-8")
print(f"\n Export CSV : ({df_15.shape[0]} lignes)")

In [ ]:
df_15

## Fusion

In [ ]:
# concaténation de df_15 et df_16 (ou lecture depuis CSV si nécessaire)

# # si déjà en mémoire : utiliser df_15, df_16 ; sinon :
# import pandas as pd
# df_15 = pd.read_csv("../data/interim/extract_15.csv", encoding="utf-8")
# df_16 = pd.read_csv("../data/interim/extract_16.csv", encoding="utf-8")

# # si besoin vérifier et aligner les colonnes
# # Mais overkill ici, on est propre normalement
# cols15 = set(df_15.columns)
# cols16 = set(df_16.columns)
# for c in sorted((cols15 | cols16) - cols15):
#     df_15[c] = pd.NA
# for c in sorted((cols15 | cols16) - cols16):
#     df_16[c] = pd.NA

# concat
df_all = pd.concat([df_15, df_16], ignore_index=True, sort=False)

##############################################################
# Pourrait en fait virer toute la déduplication car ici = 0
# Je garde si évolution des données ou choix clé déduplication
##############################################################

# TODO: déduplication ?
# choisir la clé la plus pertinente (UID + id_syceron?)
# en réalité encore des choses qui ont double entrée pour même ID_paragraphe
# mais avec texte différent = des didascalies, texte italique, etc.
# si pas de Texte, 370 lignes supprimées (qui vireraient sans doute au cleaning des données)
dup_key = ["UID", "id_syceron", "texte"]

# mask pour les lignes qui seraient supprimées par drop_duplicates
mask_removed = df_all.duplicated(subset=dup_key, keep="first")

if mask_removed.sum() == 0:
    print("Pas de doublons avec les clés choisies")
    print(f"concat {len(df_15)} + {len(df_16)} -> {len(df_all)} lignes")
else:
    removed = df_all[mask_removed].copy()
    mask_any = df_all.duplicated(subset=dup_key, keep=False)
    dupe_groups = df_all[mask_any].sort_values(by=dup_key)
    kept_in_groups = df_all[~mask_removed & mask_any]

    print(
        f"Groupes dupliqués distincts : {dupe_groups[dup_key].drop_duplicates().shape[0]}"
    )
    print(f"Lignes supprimées prévues : {len(removed)}")
    display(removed.head())
    display(dupe_groups.head())

    df_all_before = len(df_all)
    df_all = df_all.drop_duplicates(subset=dup_key, keep="first")
    print(
        f"concat {len(df_15)} + {len(df_16)} -> {df_all_before} lignes ; après déduplication {len(df_all)} lignes"
    )

# export
df_all.to_csv("../data/interim/extract_15_16_concat.csv", index=False, encoding="utf-8")
print(f"\n Export CSV : ({df_all.shape[0]} lignes)")

In [ ]:
df_all